# ShortsMaker: AI clips with MiniMax H3

Makes the AI video clips for one Short from the `veo.zip` that ShortsMaker puts in the Short's `veo` folder.

**Before you start**
1. Use the Google account **gutierrezdishann@gmail.com** (it has Colab Pro). Other accounts only get free GPUs, which are too small for H3.
2. Menu **Runtime > Change runtime type**: pick **A100 GPU** and turn on **High-RAM**, then Save.
3. Run the cells **top to bottom** (click each one's play button and wait for it to finish).

The first cell downloads about 70 GB of model files. That takes several minutes, and again in every new Colab session (Colab wipes its disk when the session ends). For the next Short in the same session, just run cells 3, 4 and 5 again (no new download).

If a cell shows an error, copy the red text to Claude.

In [ ]:
# 1. Install ComfyUI and download MiniMax H3 (skips anything already here in this session)
import os, shutil, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none")
if "A100" not in gpu:
    print("WARNING: this is not an A100. Runtime > Change runtime type > A100 GPU (needs the Pro account).")
elif "40GB" in gpu or "40 GB" in gpu or "40960" in gpu or ("80GB" not in gpu and "80 GB" not in gpu and "81920" not in gpu and "81251" not in gpu):
    print("WARNING: this is a 40 GB A100 (not 80 GB). It will run slower because weights swap with system RAM. Reconnect to try for an 80 GB card.")
!if [ -d /content/ComfyUI ]; then git -C /content/ComfyUI pull -q; else git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI; fi
!pip install -q -r /content/ComfyUI/requirements.txt hf_transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download

# Optional: paste your Hugging Face token here if rate-limited, or add it in Colab's Secrets (key icon) as HF_TOKEN
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
except Exception:
    pass
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

MODELS = "/content/ComfyUI/models"
WEIGHTS = [  # (Hugging Face repo, file in the repo, ComfyUI models folder)
    ("Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_fl2va_pruned_bf16.safetensors", "diffusion_models"),
    ("Comfy-Org/MiniMax-H3", "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors", "text_encoders"),
    ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_fp16.safetensors", "vae"),
    ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", "vae"),
    ("lightx2v/Minimax-h3-Turbo", "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", "loras"),
]
for repo, path, folder in WEIGHTS:
    name = path.split("/")[-1]
    dest = f"{MODELS}/{folder}/{name}"
    if os.path.exists(dest):
        print("already here:", name)
        continue
    print("downloading", name, "...", flush=True)
    os.makedirs(f"{MODELS}/{folder}", exist_ok=True)
    shutil.move(hf_hub_download(repo, path, local_dir="/content/hf"), dest)
print("Setup done.")

In [ ]:
# 2. Start ComfyUI in the background (it runs H3; nothing to click)
import subprocess, time, urllib.request
def comfy_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/object_info", timeout=5)
        return True
    except Exception:
        return False

if comfy_up():
    print("ComfyUI is already running.")
else:
    log = open("/content/comfyui.log", "w")
    comfy = subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
                             cwd="/content/ComfyUI", stdout=log, stderr=subprocess.STDOUT)
    for _ in range(120):  # up to 10 minutes
        if comfy_up() or comfy.poll() is not None:
            break
        time.sleep(5)
    if comfy_up():
        print("ComfyUI is ready.")
    else:
        print("ComfyUI didn't start. Last lines of its log:")
        print("".join(open("/content/comfyui.log").readlines()[-30:]))

In [ ]:
# 3. Upload veo.zip from the Short's veo folder (once per Short)
import os, shutil, zipfile
from google.colab import files
os.chdir("/content")
up = files.upload()
if not up:
    print("Upload cancelled. Run this cell again and choose veo.zip.")
else:
    shutil.rmtree("/content/job", ignore_errors=True)
    zipfile.ZipFile(next(iter(up))).extractall("/content/job")
    print(open("/content/job/jobs.json").read())

In [ ]:
# 4. Make the clips (a few minutes each; progress prints below). Change the settings and run this cell again as
#    often as you like: clips already made stay in clips.zip.
ONLY = []     # scene numbers to make, e.g. [1] for just the hook; [] makes every scene in veo.zip
SEED = 1      # same seed + same prompt gives the same clip; change it to get a different take
TURBO = True  # True: 8-step turbo (fewer compute units, faster); False: full 20 steps (best quality)

import importlib, os, sys
os.chdir("/content/job")
sys.path.insert(0, "/content/job")
import h3_run
importlib.reload(h3_run)  # a new veo.zip may carry a newer h3_run.py
h3_run.main(ONLY, SEED, TURBO)

In [ ]:
# 5. Download clips.zip, save it into the Short's veo folder, then press Enter in ShortsMaker's black window
from google.colab import files
files.download('clips.zip')